# Cobblemon Atlas｜Resolver 形態模型相容性回歸測試

使用正式解析器依各形態的 Resolver 精確指定模型、Poser 與貼圖；回報骨骼數、雜湊與有缺漏的資源。第一輪檢查普通烈空坐、超夢、超級暴鯉龍，**未能唯一解析的形態保留 audit**，不擅自套用相似模型。資料只寫入 `Cobblemon_Atlas/reports/resolver-validation/`；不修改網站及正式動畫。

In [ ]:
from pathlib import Path
import os,sys,shutil,subprocess,json,zipfile,time
DRIVE_ROOT=Path('/content/drive/MyDrive/Cobblemon_Atlas')
WORK_ROOT=Path('/content/cobblemon-bedrock')
SITE_ROOT=Path('/content/cobblemon-atlas')
BRANCH='feat/animation-manifest-scanner-phase1'
FORMS=['0384_rayquaza']
ACTIONS=['ground_walk']
FPS,SECONDS,SIZE,SS=12,2,224,2
print('Isolated Rayquaza test:',FORMS,ACTIONS)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
assert DRIVE_ROOT.is_dir(), f'找不到雲端資料夾 {DRIVE_ROOT}；請確認使用正確 Google 帳號'
for name in ['source_packs','manifests','renders','reports','checkpoints']:
    assert (DRIVE_ROOT/name).is_dir(), f'缺少 Drive 資料夾：{name}'
print('Google Drive 目錄確認成功:', DRIVE_ROOT)

In [ ]:
if not (SITE_ROOT/'.git').is_dir():
    subprocess.run(['git','clone','--depth','1','--filter=blob:none','--sparse','--branch',BRANCH,'https://github.com/hwh0410/cobblemon-atlas.git',str(SITE_ROOT)],check=True)
subprocess.run(['git','-C',str(SITE_ROOT),'sparse-checkout','set','tools','docs','notebooks','mega-showdown/data'],check=True)
subprocess.run(['git','-C',str(SITE_ROOT),'pull','--ff-only','origin',BRANCH],check=True)
sha=subprocess.check_output(['git','-C',str(SITE_ROOT),'rev-parse','--short','HEAD'],text=True).strip()
renderer_source=(SITE_ROOT/'tools/render_action_jobs.py').read_text()
assert 'def choose_actions(' in renderer_source, '目前載入的是舊版選擇器，請重新開啟 GitHub 最新 Notebook'
print('已載入最新版候選來源選擇器，Git commit:',sha)

In [ ]:
# 使用本機暫存磁碟渲染，避免每一幀反覆從 Drive 讀取大量小檔案。
WORK_ROOT.mkdir(parents=True,exist_ok=True)
packs = DRIVE_ROOT/'source_packs'
required = [packs/'development_resource_packs.zip', packs/'merged_pack.zip', packs/'render_portraits.py', DRIVE_ROOT/'manifests/animation-actions.json']
missing = [str(p) for p in required if not p.is_file()]
if missing: raise FileNotFoundError('請先上傳以下來源檔：\n'+'\n'.join(missing))
for archive in required[:2]:
    expected = WORK_ROOT/archive.stem
    if expected.is_dir():
        print('Reuse extracted pack:', expected)
        continue
    print('Extracting:', archive.name, flush=True)
    with zipfile.ZipFile(archive) as z:
        for item in z.infolist():
            dest = (WORK_ROOT/item.filename).resolve()
            if not dest.is_relative_to(WORK_ROOT.resolve()): raise ValueError('Unsafe archive path')
        z.extractall(WORK_ROOT)
    assert expected.is_dir(), f'壓縮檔最外層不是 {archive.stem}，請重新封裝'
(WORK_ROOT/'tools').mkdir(exist_ok=True)
shutil.copy2(packs/'render_portraits.py', WORK_ROOT/'tools/render_portraits.py')
manifest = DRIVE_ROOT/'manifests/animation-actions.json'
data=json.loads(manifest.read_text())
print('Manifest:',data['summary'])
print('Resources prepared locally')

In [ ]:
tool=SITE_ROOT/'tools/resolve_animation_assets.py'
tests=SITE_ROOT/'tools/test_resolve_animation_assets.py'
assert tool.is_file() and tests.is_file(),'尚未拉到新程式：請重新執行 Git pull 那格'
# Unit tests verify strict aspect matching and no silent model fallback.
subprocess.run([sys.executable,'-m','unittest','discover','-s',str(SITE_ROOT/'tools'),
                '-p','test_resolve_animation_assets.py','-v'],check=True)
out=DRIVE_ROOT/'reports/resolver-validation/resolved-forms.json'
forms=['0384_rayquaza','0150_mewtwo','0130_gyarados_mega']
cmd=[sys.executable,str(tool),'--repo',str(WORK_ROOT),'--forms',*forms,'--out',str(out)]
subprocess.run(cmd,check=True)
report=json.loads(out.read_text())
for e in report['forms']:
    item=e.get('selected') or {}
    print(e['form_id'],e['status'],e['reason'],
          'model=',item.get('model'),'bones=',item.get('bone_count'),
          'poser=',item.get('poser'))
print('Output:',out)